# The God Paradox
## Does Perfect Timing Beat Dollar-Cost Averaging?

---

### Background

Nick Maggiulli (2019) showed that a God-like investor who buys *every* market bottom
still **loses to DCA ~70% of rolling 40-year windows** when idle cash earns 0%.

This notebook replicates and extends that experiment with three God strategies,
all sharing the same setup:

| Parameter | Value |
|-----------|-------|
| Monthly contribution | $100 (DCA and God alike) |
| God's idle cash yield | 1-yr T-bill rate (Shiller pre-1962, FRED DGS1 1962-2026) |
| Dividend reinvestment | Monthly DRIP on invested shares |
| Prices | Nominal S&P 500 (1928-2026) |

### Three God Strategies

| Strategy | Rule |
|----------|------|
| **Maggiulli** | Buys at every inter-ATH trough (perfect foresight, ~98 buys) |
| **God 5-Year** | Buys once per 5-year calendar period at the period low (~21 buys) |
| **God 10-Year** | Buys once per decade at the decade low (~11 buys) |

### Two Experiments

| # | Experiment | Scope |
|---|------------|-------|
| **A** | Full run | Jan 1928 – May 2026 |
| **B** | Exhaustive rolling windows | All ~702 valid 40-year windows |


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

from god_paradox import (
    load_prices, load_dividends, load_bond_yields,
    run_all_strategies,
    NOMINAL_CSV, CHARTS_DIR,
)
from rolling_windows import (
    run_all_windows_three_strategies,
    print_summary,
    plot_windows_comparison,
)
from charts import plot_buy_timeline, plot_comparison_growth, plot_final_bars

from IPython.display import Image, display as ipy_display
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print("Loading data…")
prices      = load_prices(NOMINAL_CSV)
div_yields  = load_dividends()
bond_yields = load_bond_yields(prices)
print(f"  {len(prices)} monthly observations  "
      f"({prices.index[0].strftime('%b %Y')} – {prices.index[-1].strftime('%b %Y')})")


---
## Scenario A — Full Run (1928–2026)

All three God strategies are run simultaneously on the same nominal price series.
Every strategy uses the same $100/month contributions, T-bill cash yield, and DRIP dividends.


In [ ]:
result = run_all_strategies(prices, div_yields, bond_yields)


### When Does God Buy?

Three stacked panels — one per strategy. Each panel overlays:
- **Gray line**: S&P 500 price (log scale)
- **Red stems**: cash deployed at each buy date (stem height = dollar amount)

Labels show the cash amount for strategies with ≤ 25 buys.


In [ ]:
plot_buy_timeline(result)
ipy_display(Image(str(CHARTS_DIR / 'buy_timeline.png')))


### Portfolio Growth — All Three Strategies vs DCA


In [ ]:
plot_comparison_growth(result)
ipy_display(Image(str(CHARTS_DIR / 'comparison_growth.png')))


### Final Portfolio Values — 1928–2026


In [ ]:
plot_final_bars(result)
ipy_display(Image(str(CHARTS_DIR / 'final_bars.png')))


In [ ]:
dca_v = result['final_dca']
rows = [
    {"Strategy": "DCA",
     "Final Value": f'${dca_v:,.0f}',
     "vs DCA": "—",
     "# Buys": "every month"},
    {"Strategy": "God Maggiulli",
     "Final Value": f'${result["final_mag"]:,.0f}',
     "vs DCA": f'{result["adv_mag"]:+.1f}%',
     "# Buys": len(result['buys_mag'])},
    {"Strategy": "God 5-Year",
     "Final Value": f'${result["final_5yr"]:,.0f}',
     "vs DCA": f'{result["adv_5yr"]:+.1f}%',
     "# Buys": len(result['buys_5yr'])},
    {"Strategy": "God 10-Year",
     "Final Value": f'${result["final_10yr"]:,.0f}',
     "vs DCA": f'{result["adv_10yr"]:+.1f}%',
     "# Buys": len(result['buys_10yr'])},
]
ipy_display(pd.DataFrame(rows).set_index("Strategy"))


---
## Scenario B — Exhaustive Rolling Windows

Every valid monthly-start 40-year window in the 1928–2026 dataset is tested — **~702 windows,
no sampling bias**. This directly replicates Maggiulli's window coverage while applying all
three God strategies and T-bill cash yields.


In [ ]:
print("Running exhaustive rolling windows for all three strategies…")
print("(~2,100 simulations — takes 1–2 minutes)")
w_mag, w_5yr, w_10yr = run_all_windows_three_strategies(prices, div_yields, bond_yields)
print("Done.")


In [ ]:
print_summary(w_mag,  "GOD MAGGIULLI — ALL WINDOWS")
print_summary(w_5yr,  "GOD 5-YEAR   — ALL WINDOWS")
print_summary(w_10yr, "GOD 10-YEAR  — ALL WINDOWS")


### Rolling Window Comparison Chart

Four panels:
1. **Scatter**: God's advantage (%) vs window start year — all 3 strategies overlaid
2. **Histogram**: distribution of God's advantage — all 3 overlaid (dashed lines = medians)
3. **Grouped bars**: God win rate by starting decade for each strategy
4. **Scorecard**: summary statistics table


In [ ]:
plot_windows_comparison(w_mag, w_5yr, w_10yr)
ipy_display(Image(str(CHARTS_DIR / 'rolling_comparison.png')))


In [ ]:
def _row(name, results):
    adv = [r.god_advantage for r in results]
    gw  = sum(r.god_wins for r in results)
    return {
        "Strategy":      name,
        "Windows":       len(results),
        "God Win Rate":  f'{gw / len(results) * 100:.0f}%  ({gw}/{len(results)})',
        "Median Adv":    f'{np.median(adv):+.1f}%',
        "Mean Adv":      f'{np.mean(adv):+.1f}%',
        "Std Dev":       f'{np.std(adv):.1f}%',
        "Best (God)":    f'{max(adv):+.1f}%',
        "Worst (DCA)":   f'{min(adv):+.1f}%',
    }

ipy_display(pd.DataFrame([
    _row("Maggiulli",  w_mag),
    _row("5-Year",     w_5yr),
    _row("10-Year",    w_10yr),
]).set_index("Strategy"))


---
## Key Findings

### Full Run (1928–2026)

| Strategy | Verdict |
|----------|---------|
| **God 5-Year** | Wins by **+50.8%** — catches every 5-yr cycle crash with meaningful lump sums |
| **God 10-Year** | Wins by **+44.0%** — fewer but larger lump-sum deployments |
| **God Maggiulli** | **Loses by -24.1%** — frequent buying fragments the lump-sum advantage; most buys are under $1,000 |

### Rolling Windows (702 × 40-year periods)

| Strategy | Win Rate | Character |
|----------|----------|-----------|
| **Maggiulli** | **77%** | Tight distribution (std 9.6%) — high win rate, moderate upside, occasional severe loss |
| **5-Year** | 70% | Wide distribution (std 15%) — high upside (+51%), manageable downside (-18%) |
| **10-Year** | 52% | Widest distribution (std 18.5%) — near coin-flip, large swings both ways |

### The Core Insight

T-bill cash yield helps God most when cash **accumulates for years**. Maggiulli's frequent buying
destroys this: 90 of 98 buys deploy under $2,000 (one or two months of savings). The 5-year
strategy strikes the best balance — enough frequency to catch major crashes, enough patience to
build meaningful lump sums.
